In [1]:
import pitayaconfig
# Import the main Pyrpl class to control the Red Pitaya .
from pyrpl import Pyrpl
import auxclasses
from PyQt6.QtWidgets import QApplication
import sys

CONFIG_NAME = "Axial_mode"
# Create a Pyrpl object to connect to the Red Pitaya using the defined parameters .
p = Pyrpl(config = CONFIG_NAME, gui = False, hostname = pitayaconfig.HOSTNAME)
rp = p.redpitaya # Access the low - level RedPitaya object, .redpitaya and rp are aliases of one another.
app = QApplication(sys.argv)


: 

In [ ]:
# --- Lock - in Amplifier ( IQ Module ) Settings ---
FREQUENCY = 1000 # The expected resonant frequency of the levitated particle in Hertz ( Hz ). i.e the drive signal?
FILTER_BANDWIDTH_HZ = 1000 # The band - pass filter bandwidth in Hz. i.e. want to remove components below this frequency?
OUTPUT_GAIN = 0 # The gain applied to the demodulated signal before output. units?
DEMODULATION_PHASE_DEG = 270 # The phase shift in degrees applied to the reference oscillator. negative of velocity (-dx/dt?)
AC_COUPLING_BANDWIDTH_HZ = 100 # The bandwidth of the AC coupling filter applied to the input. This is used to removes DC and out of band
INTERNAL_OSCILLATOR_AMPLITUDE = 0.0 # The amplitude of the IQ module's internal signal generator. use 0 for feedback generator; don't want to inject any additional signal.
QUADRATURE_FACTOR = 0 # The quadrature factor for the I / Q demodulation module. Not used for feedback cooling, set to 0.

# ---- I/O settings ----
# Determines which signal is sent to the module's output_signal multiplexer to be used with other modules.  uses the raw demodulated signal, while 'quadrature' would give the low pass filtered demodulated signal or
QUAD_OR_DIRECT = 'output_direct' #'output_direct'

DIRECT_OUTPUT_ROUTING = 'out1' # The output signal routing for the I / Q demodulation module . This can be ' off ', 'out1', 'out2', or 'both', 'output_direct', 'quadrature'
INPUT_CHANNEL = 'in1' # Choose the physical input channel ( ' in1 ' or ' in2 ') connected to the detector .

In [ ]:

iq = rp.iq0 # type: ignore # Select the specified I / Q demodulation module from the list of available modules.

# Configure the I / Q demodulation module using the general parameters defined above.
# This setup creates a real - time filter to isolate the particle 's velocity signal. this is basically just doing fourier analysis...
iq.setup(frequency = FREQUENCY,
         bandwidth = FILTER_BANDWIDTH_HZ,
         gain = OUTPUT_GAIN,
         phase = DEMODULATION_PHASE_DEG,
         acbandwidth = AC_COUPLING_BANDWIDTH_HZ,
         amplitude = INTERNAL_OSCILLATOR_AMPLITUDE,
         input = INPUT_CHANNEL,
         output_direct = DIRECT_OUTPUT_ROUTING,
         output_signal = QUAD_OR_DIRECT,
         quadrature_factor = QUADRATURE_FACTOR,
         modulation_at_2f = 'off',
         demodulation_at_2f = 'on')

# If I'm just taking the delayed oscillation and feeding it back in, perhaps it's enough?
# Read the output of another 


In [ ]:
# Run the IQ demodulator control loop
iqcontrol = auxclasses.IQDemodulatorControl(rp=rp,
                                            iq_channel=rp.iq0, # type: ignore
                                            scope_monitor=monitor,
                                            quadrature_factor=1,
                                            output_direct='out2',
                                            output_signal='quadrature',
                                            input_channel ='in2')
iqcontrol.show()

In [ ]:
# PID loop that uses the output of the iq filter as input for feedback control..
# Unfinished.
PIDoutput = 'out1'
PIDin = 'iq0' # Use input of iq0..
# What is the basis of signals that the PID can apply to this?
pid = rp.pid0 # type: ignore
rp.pid0.setup(input='iq0', # type: ignore
             output_direct='out2',  # output to send pid to... should be routed internally however.
             setpoint=0.05, # pid setpoint of 50 mV - target for input..
             p=0.1,  # proportional gain factor of 0.1
             i=100,  # integrator unity-gain-frequency - the bandwidth at which the integral is calculated for
             input_filter = [3e3, 10e3])  # add 2 low-passes (3 and 10 kHz)
pid.input = 'iq0'
pid.output = 'out1'

In [ ]:

# Control an AC sine wave on one of the outputs
ACoutput = 'out1'
ACcontrol = auxclasses.ASGControlAC(rp,output=ACoutput,asgchannel=rp.asg0) # type: ignore
ACcontrol.show()

In [ ]:
# Also set other output to constant DC signal...
DCoutput = 'out2'
DCcontrol = auxclasses.DCBiasControl(rp,output=DCoutput,asgchannel=rp.asg1) # type: ignore
DCcontrol.show()
# Allowed monitors are in1, in2, asg0, asg1, pid0, pid1, iq0, but they need to be actively outputting things?

In [ ]:
sys.exit(app.exec_()) # close the QT app interface
# Close all th